# Exploración SPlink con datos de costos de atención de pacientes de COVID-19

Esta es una exploración de la biblioteca Splink siguiendo el [tutorial](https://moj-analytical-services.github.io/splink/demos/tutorials/00_Tutorial_Introduction.html) que provee la documentación y usando la base de datos:

INER_COVID19_CostoPacientes_Econo.csv


In [1]:
# Llamado de dependencias y carga de datos
import os
import pandas as pd

ruta_datos = os.path.join("data", "INER_COVID19_CostoPacientes_Econo.csv")
df = pd.read_csv(ruta_datos, encoding="utf-8")

In [2]:
df.head(5)

,EXP,NOMBRE_DEL_PACIENTE,SEXO,EDAD,GRUPO_EDAD,RESULTADO,ETIQUETAS_COVID,MOTIVO_DE_EGRESO,FECHA_INGRESO_INER,FECHA_DE_ALTA_MEJORIA,...,TOTAL_DE_EGRESOS,ESCOLARIDAD,OCUPACION,DERECHOHABIENTE_Y/O_BENEFICIARIO,VULNERABILIDAD_SOCIOECONOMICA,NIVEL_SOCIOECONOMICO,ESTADO_RESIDENCIA,CLAVE_GEOESTADISTICA_ESTATAL,MUNICIPIO_RESIDENCIA,CLAVE_GEOESTADISTICA_MUNICIPAL
0,248230,RUIZ MARTINEZ HILARIO WILFRIDO,M,76.0,75-79,SARS-COV2,COVID+RVCOVID,ALTA POR MEJORIA,2023-01-07,2023-01-18,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,TLALPAN,12.0
1,248237,MERCADO MAGDALENO LAURA ANGELICA,F,54.0,50-54,SARS-COV2,COVID+NOVACOVID,ALTA POR MEJORIA,2023-01-01,2023-01-06,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,MIGUEL HIDALGO,16.0
2,221409,GOMEZ BARRON OCTAVIO,M,41.0,40-44,SARS-COV2,COVID+NOVACOVID,ALTA POR MEJORIA,2023-01-03,2023-01-11,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,IZTAPALAPA,7.0
3,248267,DE LA ROSA RUIZ MARIA LUISA,F,86.0,85+,SARS-COV2,COVID+NOVACOVID,ALTA POR MEJORIA,2023-01-04,2023-01-12,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,LA MAGDALENA CONTRERAS,8.0
4,169798,ALMARAZ RODRIGUEZ JUANA,F,69.0,65-69,SARS-COV2,COVID+NOVACOVID,ALTA POR MEJORIA,2023-01-04,2023-01-12,...,NaN,NaN,NaN,NINGUNO,False,NaN,CIUDAD DE MEXICO,9.0,VENUSTIANO CARRANZA,17.0


In [3]:
len(df)

4632

## Preliminares

Splink requiere bases de datos lo más limpias posibles. En este caso se añade un identificador único consecutivo (`unique_id`) a la tabla, además de que se limpian espacios extras en blanco al inicio, fin y entre cada nombre completo de la columna `nombre`.

In [4]:
# Añadiendo un ID
df["unique_id"] = range(1, len(df) + 1)
df.insert(0, "unique_id", df.pop("unique_id"))

In [5]:
# Limpiando espacios en la columna nombre
df["NOMBRE_DEL_PACIENTE"] = df["NOMBRE_DEL_PACIENTE"].str.replace(r"\s+", " ", regex=True).str.strip()

## 2. Análisis Exploratorio de Datos usando Splink




Splink provee de herramientas para el análisis exploratorio de datos que facilitarán la elección de reglas de bloqueo en los siguientes pasos. Exploro aquí las que se usan en el tutorial de la biblioteca.

### Datos faltantes


In [6]:
from splink.exploratory import completeness_chart
from splink import DuckDBAPI
db_api = DuckDBAPI()
completeness_chart(df, db_api=db_api)

alt.LayerChart(...)

### Distribución de los valores en los datos


In [7]:
from splink.exploratory import profile_columns

profile_columns(df, db_api=DuckDBAPI(), top_n=10, bottom_n=5)

alt.VConcatChart(...)

## 3. Escoger las reglas de bloqueo

Se usan las reglas de bloqueo para generar pares de registros candidatos a comparar. Según la [documentación el objetivo de éstas es doble](https://moj-analytical-services.github.io/splink/demos/tutorials/03_Blocking.html#devising-effective-blocking-rules-for-prediction):

1. Eliminar suficientes pares de comparación que no coincidan para que el proceso de vinculación de registros sea lo suficientemente pequeño para que pueda calcularse.

2. Eliminar la menor cantidad posible de pares que coincidan realmente (idealmente ninguno).

Splink recomienda la generación de múltiples reglas de bloqueo para lograr ambos objetivos, por lo cual podrían ser entre 3 y 10 reglas de bloqueo.

En el análisis exploratorio del segundo paso, se observa que hay un nombre de un paciente que es el registro más repetido tanto en la columna `EXP`como en la columna `NOMBRE_DEL_PACIENTE` y que entonces estos siete registros podrían ser una sola persona.

Entonces los candidatos a las reglas de bloqueo para deduplicar estos cinco registros y demás que se repitan pueden ser:

1. `EXP` y `NOMBRE_DEL_PACIENTE`
2. `NOMBRE_DEL_PACIENTE` y `SEXO`
2. `NOMBRE_DEL_PACIENTE` y `MUNICIPIO_RESIDENCIA`


In [8]:
# Candidatos a reglas de bloqueo
from splink import block_on
block_on("EXP", "NOMBRE_DEL_PACIENTE")
block_on("NOMBRE_DEL_PACIENTE", "SEXO")
block_on("NOMBRE_DEL_PACIENTE", "MUNICIPIO_RESIDENCIA")

Para saber si los candidatos a las reglas de bloqueo funcionarán, Splink tiene diveras herramientas para ayudar a elegir.

### Contar el número de comparaciones creadas por una sóla regla de bloqueo

El número de comparaciones que genera una regla de bloqueo crece de forma cuadrática respecto al número de registros. El número de pares en una [link text](https://)tabla con N registros está dado por N(N-1)/2

In [9]:
from splink.blocking_analysis import count_comparisons_from_blocking_rule

db_api = DuckDBAPI()

br_nombre = block_on("EXP", "NOMBRE_DEL_PACIENTE")

counts = count_comparisons_from_blocking_rule(
    table_or_tables=df,
    blocking_rule=br_nombre,
    link_type="dedupe_only",
    db_api=db_api
)

counts

{'number_of_comparisons_generated_pre_filter_conditions': 5120,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 284,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."EXP" = r."EXP" AND l."NOMBRE_DEL_PACIENTE" = r."NOMBRE_DEL_PACIENTE"',
 'link_type_join_condition': 'where l."unique_id" < r."unique_id"'}

In [10]:
br_genero = block_on("NOMBRE_DEL_PACIENTE", "SEXO")

counts = count_comparisons_from_blocking_rule(
    table_or_tables=df,
    blocking_rule=br_genero,
    link_type="dedupe_only",
    db_api=db_api,
)

counts

{'number_of_comparisons_generated_pre_filter_conditions': 5208,
 'number_of_comparisons_to_be_scored_post_filter_conditions': 288,
 'filter_conditions_identified': '',
 'equi_join_conditions_identified': 'l."NOMBRE_DEL_PACIENTE" = r."NOMBRE_DEL_PACIENTE" AND l."SEXO" = r."SEXO"',
 'link_type_join_condition': 'where l."unique_id" < r."unique_id"'}

### _Worst offending values_

In [11]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    table_or_tables=df,
    blocking_rule= block_on("EXP", "NOMBRE_DEL_PACIENTE"),
    link_type="dedupe_only",
    db_api=db_api,
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,count_l,count_r,block_count
0,239316,MENESES AGUAS GLENNY EDUARDO,7,7,49
1,242300,VALDEZ CHAPARRO ELIZABETH,5,5,25
2,240022,JIMENEZ MARTINEZ MIGUEL ANGEL,4,4,16
3,237605,IBARRA LOPEZ NOE,4,4,16
4,238904,HERNANDEZ LOPEZ CRECENCIO,4,4,16
5,229037,MORENO AGUILAR DIANA LUISA,4,4,16
6,241300,CRUZ RODRIGUEZ ANGELA ISABEL,4,4,16
7,114639,LEMUS HERNANDEZ GABINO,4,4,16
8,237871,GONZALEZ LARA JOSE EMILIANO,4,4,16
9,242581,COLLADO PEÑA SUSANA PATRICIA,4,4,16


In [12]:
from splink.blocking_analysis import n_largest_blocks

result = n_largest_blocks(    table_or_tables=df,
    blocking_rule= block_on("NOMBRE_DEL_PACIENTE", "SEXO"),
    link_type="dedupe_only",
    db_api=db_api,
    n_largest=10
    )

result.as_pandas_dataframe()

,key_0,key_1,count_l,count_r,block_count
0,MENESES AGUAS GLENNY EDUARDO,M,7,7,49
1,VALDEZ CHAPARRO ELIZABETH,F,5,5,25
2,COLLADO PEÑA SUSANA PATRICIA,F,4,4,16
3,GONZALEZ LARA JOSE EMILIANO,M,4,4,16
4,JIMENEZ MARTINEZ MIGUEL ANGEL,M,4,4,16
5,CRUZ RODRIGUEZ ANGELA ISABEL,F,4,4,16
6,IBARRA LOPEZ NOE,M,4,4,16
7,LEMUS HERNANDEZ GABINO,M,4,4,16
8,MORENO AGUILAR DIANA LUISA,F,4,4,16
9,HERNANDEZ LOPEZ CRECENCIO,M,4,4,16


### Contar el número de comparaciones creadas por una lista de reglas de agrupamiento

Genera una gráfica de comparaciones acumuladas donde cada barra representa cuántas comparaciones nuevas y únicas aporta cada regla adicional, después de desduplicar contra las reglas anteriores.

- Si una regla aporta muy pocas comparaciones nuevas y únicas significa que casi todos los pares de registros nuevos que genera ya estaban cubiertos por alguna regla o reglas anteriores, es decir, es redundante.
- Si una regla aporta muchas más comparaciones nuevas y únicas significa que está capturando pares que las otras reglas no cubren, por lo tanto, es valiiosa.
- El acumulado final me indica si la lista de reglas es manejable computacionalmente.

In [13]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

blocking_rules_for_analysis = [
    block_on("EXP", "NOMBRE_DEL_PACIENTE"),
    block_on("NOMBRE_DEL_PACIENTE", "SEXO"),
    block_on("NOMBRE_DEL_PACIENTE", "MUNICIPIO_RESIDENCIA")
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df,
    blocking_rules=blocking_rules_for_analysis,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

De esta gráfica podemos concluir que mis reglas de bloqueo pueden ser un poco redundantes, ya que el tamaño de la segunda y tercer barras es pequeño, es decir no se generan nuevos pares a comparar. Lo ideal es elegir reglas de agrupamiento que generen más pares a comparar.

## 4. Estimar los parámetros del modelo

En este punto voy a modificar un poco mis reglas de bloqueo y agregar fecha de ingreso y egreso:

In [14]:
# Candidatos a reglas de bloqueo
from splink import block_on
block_on("EXP", "NOMBRE_DEL_PACIENTE")
block_on("NOMBRE_DEL_PACIENTE", "SEXO")
block_on("NOMBRE_DEL_PACIENTE", "MUNICIPIO_RESIDENCIA")
block_on("NOMBRE_DEL_PACIENTE", "FECHA_INGRESO_INER")
block_on("NOMBRE_DEL_PACIENTE", "FECHA_ALTA_MEJORIA")

In [15]:
# Con este nuevo bloque
blocking_rules_for_analysis = [
block_on("EXP", "NOMBRE_DEL_PACIENTE"),
block_on("NOMBRE_DEL_PACIENTE", "SEXO"),
block_on("NOMBRE_DEL_PACIENTE", "MUNICIPIO_RESIDENCIA"),
block_on("NOMBRE_DEL_PACIENTE", "FECHA_INGRESO_INER"),
block_on("NOMBRE_DEL_PACIENTE", "FECHA_DE_ALTA_MEJORIA")
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=df,
    blocking_rules=blocking_rules_for_analysis,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

### Comparaciones (`Comparison`)

Una Comparación o *Comparison* representa como se va a evaluar la similitud de un campo. El modelo que se construye usando Splink consta de muchas comparaciones.

Las comparaciones tienen niveles `ComparisonLevels` donde se asignan calificaciones de similitud entre las columnas para cierta comparación. Por ejemplo:




```
Modelo de vinculación de datos
├─-- Comparison: fechaing
│    ├─-- ComparisonLevel: Coincidencia exacta
│    ├─-- ComparisonLevel: Un caracter de diferencia
│    ├─-- ComparisonLevel: Cualquier otra
├─-- Comparison: nombre
│    ├─-- ComparisonLevel: Coincidencia exacta
│    ├─-- ComparisonLevel: JaroWinkler > 0.9
│    ├─-- ComparisonLevel: Cualquier otra
│    etc.
```



Para `fechaing` o pienso que en particular para cualquier fecha
| fechaing_l      | fechaegr_r      | comparison_level        | interpretation |
|------------|------------|--------------------------|-----------------|
| 1971-05-24 | 1971-05-24 | Coincidencia exacta            | great match     |
| 1971-05-24 | 1971-06-24 | Un caracter de diferencia | fuzzy match     |
| 1971-05-24 | 2000-01-02 | Cualquier otra                | bad match       |

Para `nombre`

| nombre_l | nombre_r | comparison_level | interpretation                                    |
|-----------|-----------|-------------------|----------------------------------------------------|
| VICENTE MARTIN VALENCIA CHAVEZ       | VICENTE MARTIN VALENCIA CHAVEZ       | Exact match       | great match |
| MARTIN VALENCIA CHAVEZ       | VICENTE MARTIN VALENCIA CHAVEZ    | All JaroWinkler >0.9         | great match?  
| VICENTE MARTIN VLENCIA CHAVEZ       | JOSE PEDRO VALENCIA CHAVEZ    | other         | bad match, this comparison has no notion of nicknames

En el segundo caso de nombre donde difiere por el segundo nombre el registro, no es tan conveniente usar `JaroWinkler`, ya que este método le da peso al inicio del string y al faltar el primer nombre de Vicente, la comparación ya no es tan ideal. Para ello Splink tiene un módulo de comparaciones "out of the box" que incluyen estos casos, así como también se pueden customizar las reglas para hacer comparaciones.

### Especificar el modelo usando las comparaciones

Categoría 1: Funciones genéricas que aplican una función de fuzzy matching en particular. Por ejemplo, distancia de Levenshtein.

In [16]:
import splink.comparison_library as cl

nombre_comparacion = cl.LevenshteinAtThresholds("nombre", 2)
print(nombre_comparacion.get_comparison("duckdb").human_readable_description)

Comparison 'LevenshteinAtThresholds' of "nombre".
Similarity is assessed using the following ComparisonLevels:
    - 'nombre is NULL' with SQL rule: "nombre_l" IS NULL OR "nombre_r" IS NULL
    - 'Exact match on nombre' with SQL rule: "nombre_l" = "nombre_r"
    - 'Levenshtein distance of nombre <= 2' with SQL rule: levenshtein("nombre_l", "nombre_r") <= 2
    - 'All other comparisons' with SQL rule: ELSE



Categoría 2: Funciones de comparación diseñadas para tipos de datos específicos.

In [17]:
nombre_completo_comparacion = cl.NameComparison("nombre")
print(nombre_completo_comparacion.get_comparison("duckdb").human_readable_description)

Comparison 'NameComparison' of "nombre".
Similarity is assessed using the following ComparisonLevels:
    - 'nombre is NULL' with SQL rule: "nombre_l" IS NULL OR "nombre_r" IS NULL
    - 'Exact match on nombre' with SQL rule: "nombre_l" = "nombre_r"
    - 'Jaro-Winkler distance of nombre >= 0.92' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.92
    - 'Jaro-Winkler distance of nombre >= 0.88' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.88
    - 'Jaro-Winkler distance of nombre >= 0.7' with SQL rule: jaro_winkler_similarity("nombre_l", "nombre_r") >= 0.7
    - 'All other comparisons' with SQL rule: ELSE



### Especificar el diccionario de configuraciones

In [18]:
# Antes para poder comparar las fechas de ingreso y egreso voy a customizar un poco la función DateOfBirthComparison

fecha_ingreso_comparacion = cl.DateOfBirthComparison(
    "FECHA_INGRESO_INER",
    input_is_string=True,
    datetime_format="%Y-%m-%d",
    invalid_dates_as_null=True,
    datetime_thresholds=[1, 7, 365],       # límites en días
    datetime_metrics=["day", "day", "day"] # unidad de cada threshold
)

fecha_egreso_comparacion = cl.DateOfBirthComparison(
    "FECHA_DE_ALTA_MEJORIA",
    input_is_string=True,
    datetime_format="%Y-%m-%d",
    invalid_dates_as_null=True,
    datetime_thresholds=[1, 7, 365],       # límites en días
    datetime_metrics=["day", "day", "day"] # unidad de cada threshold
)

In [19]:
from splink import Linker, SettingsCreator, block_on, DuckDBAPI

settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("NOMBRE_DEL_PACIENTE"),
        fecha_ingreso_comparacion,
        fecha_egreso_comparacion
    ],
    blocking_rules_to_generate_predictions=[
    block_on("NOMBRE_DEL_PACIENTE", "FECHA_INGRESO_INER"),
    block_on("NOMBRE_DEL_PACIENTE", "FECHA_DE_ALTA_MEJORIA")
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df, settings, db_api=DuckDBAPI())

### Estimar los parámetros del modelo

a. El parámetro `probability_two_random_records_match` es la probabilidad de que dos registros tomados al azar de tus datos de entrada representen un match (típicamente un número muy pequeño).

b. Los valores `u` son la proporción de registros que caen en cada `ComparisonLevel` entre los registros que verdaderamente no son coincidencia.

c. Los valores `m` son la proporción de registros que caen en cada `ComparisonLevel` entre los registros que verdaderamente sí son coincidencia.

In [20]:
# a. probability_two_random_records_matchdf_predictions

deterministic_rules = [
    block_on("NOMBRE_DEL_PACIENTE", "EXP")
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)

Probability two random records match is estimated to be  3.78e-05.
This means that amongst all possible pairwise record comparisons, one in 26,435.84 are expected to match.  With 10,725,396 total possible comparisons, we expect a total of around 405.71 matching pairs


In [21]:
# b.
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - NOMBRE_DEL_PACIENTE (no m values are trained).
    - FECHA_INGRESO_INER (no m values are trained).
    - FECHA_DE_ALTA_MEJORIA (no m values are trained).


In [22]:
# c.
linker.training.estimate_m_from_label_column("EXP")

---------- Estimating m probabilities using from column EXP ----------
m probability not trained for NOMBRE_DEL_PACIENTE - Jaro-Winkler distance of NOMBRE_DEL_PACIENTE >= 0.92 (comparison vector value: 3). This usually means the comparison level was never observed in the training data.
m probability not trained for NOMBRE_DEL_PACIENTE - Jaro-Winkler distance of NOMBRE_DEL_PACIENTE >= 0.88 (comparison vector value: 2). This usually means the comparison level was never observed in the training data.
m probability not trained for NOMBRE_DEL_PACIENTE - Jaro-Winkler distance of NOMBRE_DEL_PACIENTE >= 0.7 (comparison vector value: 1). This usually means the comparison level was never observed in the training data.
m probability not trained for FECHA_INGRESO_INER - Abs date difference <= 1 day (comparison vector value: 3). This usually means the comparison level was never observed in the training data.
m probability not trained for FECHA_DE_ALTA_MEJORIA - Abs date difference <= 1 day (compari

In [23]:
#c.
sesion_2 = block_on("NOMBRE_DEL_PACIENTE")
linker.training.estimate_parameters_using_expectation_maximisation(sesion_2)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."NOMBRE_DEL_PACIENTE" = r."NOMBRE_DEL_PACIENTE"

Parameter estimates will be made for the following comparison(s):
    - FECHA_INGRESO_INER
    - FECHA_DE_ALTA_MEJORIA

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - NOMBRE_DEL_PACIENTE

Level Abs date difference <= 1 day on comparison FECHA_INGRESO_INER not observed in dataset, unable to train m value

Level Abs date difference <= 1 day on comparison FECHA_DE_ALTA_MEJORIA not observed in dataset, unable to train m value

Iteration 1: Largest change in params was 0.775 in the m_probability of FECHA_INGRESO_INER, level `Exact match on date of birth`
Iteration 2: Largest change in params was 0.19 in the m_probability of FECHA_DE_ALTA_MEJORIA, level `Exact match on date of birth`
Iteration 3: Largest change in params was 0.314 in the m_probability of FECHA_DE_ALTA_MEJ

<EMTrainingSession, blocking on l."NOMBRE_DEL_PACIENTE" = r."NOMBRE_DEL_PACIENTE", deactivating comparisons NOMBRE_DEL_PACIENTE>

### Visualizando los pesos

In [24]:
linker.visualisations.match_weights_chart()

/home/pradel/Desktop/05_utumno/00_lentigob/lentigob-splink/exploracion/.venv/lib/python3.10/site-packages/altair/vegalite/v6/api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [25]:
linker.visualisations.m_u_parameters_chart()


alt.HConcatChart(...)

## 5. Predicción

In [26]:
df_predictions = linker.inference.predict(threshold_match_probability=0.2)
df_predictions.as_pandas_dataframe(limit=25)

Blocking time: 0.01 seconds
Predict time: 0.09 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'NOMBRE_DEL_PACIENTE':
    m values not fully trained
Comparison: 'FECHA_INGRESO_INER':
    m values not fully trained
Comparison: 'FECHA_DE_ALTA_MEJORIA':
    m values not fully trained


,match_weight,match_probability,unique_id_l,unique_id_r,NOMBRE_DEL_PACIENTE_l,NOMBRE_DEL_PACIENTE_r,gamma_NOMBRE_DEL_PACIENTE,tf_NOMBRE_DEL_PACIENTE_l,tf_NOMBRE_DEL_PACIENTE_r,bf_NOMBRE_DEL_PACIENTE,bf_tf_adj_NOMBRE_DEL_PACIENTE,FECHA_INGRESO_INER_l,FECHA_INGRESO_INER_r,gamma_FECHA_INGRESO_INER,bf_FECHA_INGRESO_INER,FECHA_DE_ALTA_MEJORIA_l,FECHA_DE_ALTA_MEJORIA_r,gamma_FECHA_DE_ALTA_MEJORIA,bf_FECHA_DE_ALTA_MEJORIA,match_key
0,5.065879,0.97101,4203,4470,CLEMENTE MILLAN MARIA ALEJANDRA,CLEMENTE MILLAN MARIA ALEJANDRA,4,0.000432,0.000432,28498.69272,0.079586,2020-11-08,2020-11-08,5,372.26605,2020-11-21,2020-12-04,1,1.048687,0
1,5.065879,0.97101,3102,4552,PAZ GOMEZ JULIA,PAZ GOMEZ JULIA,4,0.000432,0.000432,28498.69272,0.079586,2020-03-28,2020-03-28,5,372.26605,2020-05-07,2020-06-11,1,1.048687,0
2,12.439584,0.99982,452,455,VILLANUEVA REYES MARTA,VILLANUEVA REYES MARTA,4,0.000432,0.000432,28498.69272,0.079586,2022-02-16,2022-02-16,5,372.26605,2022-02-24,2022-02-24,5,173.921118,0
3,12.439584,0.99982,386,457,ALVAREZ RICO GABRIEL,ALVAREZ RICO GABRIEL,4,0.000432,0.000432,28498.69272,0.079586,2022-02-11,2022-02-11,5,372.26605,2022-02-13,2022-02-13,5,173.921118,0
4,5.065879,0.97101,495,564,NAVARRO ALVAREZ JOSE ALBERTO,NAVARRO ALVAREZ JOSE ALBERTO,4,0.000432,0.000432,28498.69272,0.079586,2022-03-07,2022-03-07,5,372.26605,2022-03-08,2022-04-16,1,1.048687,0
5,12.439584,0.99982,881,882,ROMO MARTINEZ XIMENA ISABEL,ROMO MARTINEZ XIMENA ISABEL,4,0.000432,0.000432,28498.69272,0.079586,2022-12-26,2022-12-26,5,372.26605,2022-12-27,2022-12-27,5,173.921118,0
6,5.065879,0.97101,1965,2104,FLORES GUTIERREZ DIEGO GONZALO,FLORES GUTIERREZ DIEGO GONZALO,4,0.000432,0.000432,28498.69272,0.079586,2021-06-16,2021-06-16,5,372.26605,2021-06-18,2021-07-13,1,1.048687,0
7,12.439584,0.99982,2727,2778,ARGUELLO DE ROSINI ANITA ESTER,ARGUELLO DE ROSINI ANITA ESTER,4,0.000432,0.000432,28498.69272,0.079586,2021-10-16,2021-10-16,5,372.26605,2021-10-28,2021-10-28,5,173.921118,0
8,12.439584,0.99982,2882,2901,GONZALEZ MARTINEZ JUAN MANUEL,GONZALEZ MARTINEZ JUAN MANUEL,4,0.000432,0.000432,28498.69272,0.079586,2021-09-16,2021-09-16,5,372.26605,2021-11-29,2021-11-29,5,173.921118,0
9,5.065879,0.97101,3122,3373,GARCIA VAZQUEZ PATRICIA,GARCIA VAZQUEZ PATRICIA,4,0.000432,0.000432,28498.69272,0.079586,2020-05-09,2020-05-09,5,372.26605,2020-05-15,2020-06-30,1,1.048687,0
